<a href="https://colab.research.google.com/github/Kyrylo-Shyvam/erase-token/blob/jules_wip_17086835646078764891/Erasing_tokens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Markdown Cell: Analysis and Observations
# (To be placed after the "User Experimentation Playground" cell,
# and potentially referenced by the demonstration scenario cells,
# or after all demonstration cells as a concluding analysis section.)

## Analysis and Your Observations

This section is for you to reflect on the experiments you've run! After using the "User Experimentation Playground" or running the demonstration scenarios, take a moment to note down your findings. Consider the following aspects:

**1. Model Adaptation Strategies:**
*   **How do different models adapt (or fail to adapt) to specific word bans?**
    *   Do they use synonyms? (e.g., if "car" is banned, does it say "automobile"?)
    *   Do they use circumlocution or rephrasing? (e.g., describing the concept without naming it)
    *   Do they simply omit information related to the banned word?
    *   Are there noticeable differences between models (e.g., gpt2-large vs. Gemma vs. Qwen)?

**2. Impact of Random Vocabulary Banning:**
*   **What is the threshold at which responses become incoherent under random bans?**
    *   Observe the output quality as you increase the `random_ban_percentage` (e.g., 1%, 5%, 10%, 20%).
    *   At what point does the text become nonsensical or difficult to understand?
    *   Does this threshold vary by model or by prompt type?
*   **What are the typical failure modes?** (e.g., repetitive phrases, grammatical errors, loss of context, use of very unusual or archaic words)

**3. "Awareness" of Banned Terms:**
*   When a common word is banned (like "apple" in the fruit listing demo):
    *   Is the word successfully omitted?
    *   Does the model seem to struggle (e.g., listing fewer items than requested, awkward phrasing, stopping prematurely)?
    *   Does it try to "work around" the ban in clever ways?

**4. Type of Token Banned:**
*   **Are there notable differences in model behavior when banning different types of tokens?**
    *   For example, banning a common noun (e.g., "apple") vs. a verb (e.g., "run") vs. a preposition or symbol (e.g., "+").
    *   Does banning a symbol crucial for a task (like "+" in "2 + 2") completely break the task, or does the model attempt an alternative representation or refuse to answer?

**5. General Coherence and Quality:**
*   Beyond specific bans, how does the overall quality of the text change?
*   Does the model maintain context and logical flow? Is the generated text fluent and natural-sounding?

**How to Add Your Notes:**
*   You can double-click this markdown cell to edit it and add your notes directly.
*   Alternatively, you can create new markdown cells below this one (or below specific experiments) by clicking the `+ Text` button in the Colab toolbar.

*Happy experimenting and analyzing!*

# END OF CONTENT

In [ ]:
!git clone https://github.com/Kyrylo-Shyvam/erase-token.git

Cloning into 'erase-token'...
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 11 (delta 2), reused 11 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (11/11), 18.78 KiB | 18.78 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [1]:
#CELL BREAK
# Cell 1: Installs and Imports
!pip install transformers torch sentencepiece accelerate

import torch
import random
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor
import gc # For explicit garbage collection

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 33.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [2]:
class TokenBanningLogitsProcessor(LogitsProcessor):
    def __init__(self, banned_token_ids: list[int]):
        super().__init__()
        self.banned_token_ids = banned_token_ids

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        for token_id in self.banned_token_ids:
            if 0 <= token_id < scores.shape[-1]: # Check if token_id is within the vocabulary range
                scores[:, token_id] = -float('inf')
        return scores

In [3]:
def generate_response(model, tokenizer, prompt_text: str, specific_banned_words: list[str] = None, random_ban_percentage: float = 0.0, max_new_tokens: int = 50):
    if specific_banned_words is None:
        specific_banned_words = []

    print(f"--- Generating Response for model: {model.config._name_or_path} ---") # Use _name_or_path for better model ID
    print(f"Prompt: {prompt_text}")
    print(f"Specific Banned Words: {specific_banned_words}")
    print(f"Random Ban Percentage: {random_ban_percentage*100:.2f}%")
    print(f"Max New Tokens: {max_new_tokens}")

    all_banned_token_ids = []
    if specific_banned_words:
        print(f"Tokenizing specific banned words: {specific_banned_words}")
        temp_specific_ids = []
        for word in specific_banned_words:
            # Tokenize the word itself. add_special_tokens=False is important.
            token_ids = tokenizer.encode(word, add_special_tokens=False)
            if not token_ids: # Handle cases where a word might tokenize to nothing
                print(f"Warning: Word '{word}' tokenized to an empty list.")
                continue
            print(f"  '{word}' -> {token_ids}")
            temp_specific_ids.extend(token_ids)
        if temp_specific_ids:
             print(f"Specific banned words tokenized to {len(set(temp_specific_ids))} unique ID(s): {list(set(temp_specific_ids))[:20]}... (truncated)")
        all_banned_token_ids.extend(temp_specific_ids)

    if random_ban_percentage > 0.0:
        vocab_size = tokenizer.vocab_size
        num_tokens_to_ban = int(random_ban_percentage * vocab_size)

        current_banned_set = set(all_banned_token_ids)
        population_for_random = [i for i in range(vocab_size) if i not in current_banned_set]

        num_tokens_to_ban = min(num_tokens_to_ban, len(population_for_random))

        if num_tokens_to_ban > 0:
            print(f"Randomly banning {num_tokens_to_ban} tokens from {len(population_for_random)} eligible tokens (vocab size: {vocab_size}).")
            randomly_banned_ids = random.sample(population_for_random, num_tokens_to_ban)
            all_banned_token_ids.extend(randomly_banned_ids)
        else:
            print("No tokens to ban randomly (either percentage too low, vocab too small, or all eligible tokens already banned specifically).")

    if all_banned_token_ids:
        all_banned_token_ids = sorted(list(set(all_banned_token_ids))) # Ensure uniqueness and sort
        print(f"Final list of {len(all_banned_token_ids)} unique token ID(s) to ban: {all_banned_token_ids[:20]}... (truncated if >20)")
    else:
        print("No tokens will be banned for this generation.")

    logits_processors = []
    if all_banned_token_ids:
        processor = TokenBanningLogitsProcessor(banned_token_ids=all_banned_token_ids)
        logits_processors.append(processor)
        print("TokenBanningLogitsProcessor instantiated.")
    else:
        print("No banning processor needed.")

    print("Encoding prompt...")
    inputs = tokenizer(prompt_text, return_tensors='pt')
    input_ids = inputs.input_ids

    # Move inputs to the same device as the model
    if model.device.type == 'cuda': # Check if model is on GPU
        device = model.device
        input_ids = input_ids.to(device)
        print(f"Input tensors moved to device: {device}")

    print("Generating response with model.generate()...")
    output_sequences = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        logits_processor=logits_processors if logits_processors else None,
        pad_token_id=tokenizer.pad_token_id # Explicitly set pad_token_id
        # Other parameters like temperature, top_k can be added here
        # no_repeat_ngram_size=2 # Example to prevent some repetition
    )
    print("Generation complete.")

    print("Decoding response...")
    # Slice the output_sequences to get only the generated tokens
    generated_ids = output_sequences[0, input_ids.shape[-1]:]
    decoded_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    print(f"Generated text: {decoded_text}")
    print(f"--- End of Generation ---")
    return decoded_text

In [4]:
def generate_response(model, tokenizer, prompt_text: str, specific_banned_words: list[str] = None, random_ban_percentage: float = 0.0, max_new_tokens: int = 50):
    if specific_banned_words is None:
        specific_banned_words = []

    print(f"--- Generating Response for model: {model.config._name_or_path} ---") # Use _name_or_path for better model ID
    print(f"Prompt: {prompt_text}")
    print(f"Specific Banned Words: {specific_banned_words}")
    print(f"Random Ban Percentage: {random_ban_percentage*100:.2f}%")
    print(f"Max New Tokens: {max_new_tokens}")

    all_banned_token_ids = []
    if specific_banned_words:
        print(f"Tokenizing specific banned words: {specific_banned_words}")
        temp_specific_ids = []
        for word in specific_banned_words:
            # Tokenize the word itself. add_special_tokens=False is important.
            token_ids = tokenizer.encode(word, add_special_tokens=False)
            if not token_ids: # Handle cases where a word might tokenize to nothing
                print(f"Warning: Word '{word}' tokenized to an empty list.")
                continue
            print(f"  '{word}' -> {token_ids}")
            temp_specific_ids.extend(token_ids)
        if temp_specific_ids:
             print(f"Specific banned words tokenized to {len(set(temp_specific_ids))} unique ID(s): {list(set(temp_specific_ids))[:20]}... (truncated)")
        all_banned_token_ids.extend(temp_specific_ids)

    if random_ban_percentage > 0.0:
        vocab_size = tokenizer.vocab_size
        num_tokens_to_ban = int(random_ban_percentage * vocab_size)

        current_banned_set = set(all_banned_token_ids)
        population_for_random = [i for i in range(vocab_size) if i not in current_banned_set]

        num_tokens_to_ban = min(num_tokens_to_ban, len(population_for_random))

        if num_tokens_to_ban > 0:
            print(f"Randomly banning {num_tokens_to_ban} tokens from {len(population_for_random)} eligible tokens (vocab size: {vocab_size}).")
            randomly_banned_ids = random.sample(population_for_random, num_tokens_to_ban)
            all_banned_token_ids.extend(randomly_banned_ids)
        else:
            print("No tokens to ban randomly (either percentage too low, vocab too small, or all eligible tokens already banned specifically).")

    if all_banned_token_ids:
        all_banned_token_ids = sorted(list(set(all_banned_token_ids))) # Ensure uniqueness and sort
        print(f"Final list of {len(all_banned_token_ids)} unique token ID(s) to ban: {all_banned_token_ids[:20]}... (truncated if >20)")
    else:
        print("No tokens will be banned for this generation.")

    logits_processors = []
    if all_banned_token_ids:
        processor = TokenBanningLogitsProcessor(banned_token_ids=all_banned_token_ids)
        logits_processors.append(processor)
        print("TokenBanningLogitsProcessor instantiated.")
    else:
        print("No banning processor needed.")

    print("Encoding prompt...")
    inputs = tokenizer(prompt_text, return_tensors='pt')
    input_ids = inputs.input_ids

    # Move inputs to the same device as the model
    if model.device.type == 'cuda': # Check if model is on GPU
        device = model.device
        input_ids = input_ids.to(device)
        print(f"Input tensors moved to device: {device}")

    print("Generating response with model.generate()...")
    output_sequences = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        logits_processor=logits_processors if logits_processors else None,
        pad_token_id=tokenizer.pad_token_id # Explicitly set pad_token_id
        # Other parameters like temperature, top_k can be added here
        # no_repeat_ngram_size=2 # Example to prevent some repetition
    )
    print("Generation complete.")

    print("Decoding response...")
    # Slice the output_sequences to get only the generated tokens
    generated_ids = output_sequences[0, input_ids.shape[-1]:]
    decoded_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    print(f"Generated text: {decoded_text}")
    print(f"--- End of Generation ---")
    return decoded_text

In [7]:
print("=======================================================================")
print("== User Experimentation Playground ==")
print("== Modify the parameters below and re-run this cell to experiment. ==")
print("=======================================================================")

# --- 1. Configure Your Experiment ---

# Choose a model ID from the list:
# Available: "gpt2-large", "google/gemma-2b-it", "Qwen/Qwen2-1.5B-Instruct"
# (Make sure the model you select was loaded in the main execution cell, or adapt to load here)
# For simplicity, this cell will attempt to load the selected model.
# Be mindful of Colab's VRAM limits when switching models frequently.
selected_model_id = "Qwen/Qwen2.5-1.5B-Instruct"  # @param ["gpt2-large", "google/gemma-3-1b-it", "Qwen/Qwen2.5-1.5B-Instruct"]

user_prompt_text = "Describe a beautiful sunset over the ocean. It"  # @param {type:"string"}

# List of words/phrases to ban specifically. Example: ["beautiful", "ocean"]
user_specific_banned_words = ['beautiful', ' beautiful', ' day']  # @param {type:"raw"}

# Percentage of random vocabulary to ban (0.0 to 1.0). Example: 0.05 for 5%
user_random_ban_percentage = 0.0  # @param {type:"slider", min:0.0, max:1.0, step:0.01}

user_max_new_tokens = 100  # @param {type:"integer"}

# --- 2. Setup and Run Experiment ---

# (Helper function to load model and tokenizer if not already loaded)
# Note: This is a simplified loader for this cell.
# The main loop (Cell 4) has more robust loading for multiple models.
# Consider running Cell 4 first if you encounter issues here or want all models pre-downloaded.

_loaded_models_cache = {} # Basic cache for this cell

def get_model_and_tokenizer_for_experiment(model_id):
    if model_id in _loaded_models_cache:
        print(f"Using cached model and tokenizer for {model_id}.")
        return _loaded_models_cache[model_id]['model'], _loaded_models_cache[model_id]['tokenizer']

    print(f"Loading tokenizer for {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        print(f"Set tokenizer.pad_token to tokenizer.eos_token for {model_id}")

    model_args = {}
    if "gemma" in model_id.lower():
        # For Colab, bfloat16 is often good for Gemma 2B if on T4 GPU or newer
        # V100 might prefer float16. A100 supports bfloat16.
        # Using float16 as a safer default if bfloat16 is not universally available/performant
        model_args['torch_dtype'] = torch.float16
        # model_args['low_cpu_mem_usage'] = True # Already in main loop, good for Colab
        print(f"Note: For Gemma ({model_id}), torch_dtype=torch.float16 is set for this cell.")
    if "qwen" in model_id.lower():
        model_args['trust_remote_code'] = True
        print(f"Note: For Qwen ({model_id}), trust_remote_code=True is set.")

    print(f"Loading model {model_id} with args {model_args}...")
    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, **model_args)
        if torch.cuda.is_available():
            print(f"Moving model {model_id} to CUDA device.")
            model.to('cuda')
        else:
            print(f"CUDA not available for {model_id}, using CPU.")

        # Clear cache if it grows too large (e.g., more than 1 model)
        # Only keep one model loaded via this cell's cache
        if len(_loaded_models_cache) >= 1:
            print("Clearing previous model(s) from this cell's cache to save memory...")
            # Basic FIFO like, or just clear all but current
            # This doesn't handle models loaded by other cells.
            # Make a copy of keys to iterate over for deletion
            keys_to_delete = [key for key in _loaded_models_cache.keys() if key != model_id]

            for key_to_del in keys_to_delete:
                print(f"Removing {key_to_del} from this cell's cache.")
                # It's important to ensure that model and tokenizer are actually objects before del
                if 'model' in _loaded_models_cache[key_to_del] and _loaded_models_cache[key_to_del]['model'] is not None:
                    del _loaded_models_cache[key_to_del]['model']
                if 'tokenizer' in _loaded_models_cache[key_to_del] and _loaded_models_cache[key_to_del]['tokenizer'] is not None:
                    del _loaded_models_cache[key_to_del]['tokenizer']
                del _loaded_models_cache[key_to_del] # remove entry

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

        _loaded_models_cache[model_id] = {'model': model, 'tokenizer': tokenizer}
        return model, tokenizer
    except Exception as e:
        print(f"Error loading model {model_id}: {e}")
        print("Please ensure you have enough VRAM/RAM, and the model ID is correct.")
        print("You might need to restart the Colab session if memory issues persist.")
        return None, None

current_model, current_tokenizer = get_model_and_tokenizer_for_experiment(selected_model_id)

if current_model and current_tokenizer:
    print(f"\n--- Experiment Details ---")
    print(f"Model: {selected_model_id}") # Or use current_model.config._name_or_path
    print(f"Prompt: {user_prompt_text}")
    print(f"Max New Tokens: {user_max_new_tokens}")

    # Baseline Generation (No Bans)
    print(f"\n--- Generating Baseline (No Bans) ---")
    baseline_output = generate_response(
        model=current_model,
        tokenizer=current_tokenizer,
        prompt_text=user_prompt_text,
        max_new_tokens=user_max_new_tokens
    )
    print(f"\nBaseline Output:\n{user_prompt_text}{baseline_output}")

    # Generation with User-Specified Bans
    if user_specific_banned_words or user_random_ban_percentage > 0.0:
        print(f"\n--- Generating with User Bans ---")
        # print(f"Specific Banned Words: {user_specific_banned_words}") # generate_response will print this
        # print(f"Random Ban Percentage: {user_random_ban_percentage*100:.2f}%") # generate_response will print this

        banned_output = generate_response(
            model=current_model,
            tokenizer=current_tokenizer,
            prompt_text=user_prompt_text,
            specific_banned_words=user_specific_banned_words,
            random_ban_percentage=user_random_ban_percentage,
            max_new_tokens=user_max_new_tokens
        )
        print(f"\nOutput with Bans:\n{user_prompt_text}{banned_output}")
    else:
        print("\nNo bans specified by the user (specific or random). Skipping ban generation.")

    print("\n=======================================================================")
    print("== Experiment Finished. Modify parameters above and re-run. ==")
    print("=======================================================================")
else:
    print("Could not run experiment due to model loading issues. Please check errors above.")
    print("Ensure the selected model ID is correct and Colab has sufficient resources.")
# END OF CELL CONTENT

== User Experimentation Playground ==
== Modify the parameters below and re-run this cell to experiment. ==
Loading tokenizer for Qwen/Qwen2.5-1.5B-Instruct...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Moving model Qwen/Qwen2.5-1.5B-Instruct to CUDA device.


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



--- Experiment Details ---
Model: Qwen/Qwen2.5-1.5B-Instruct
Prompt: Describe a beautiful sunset over the ocean. It
Max New Tokens: 100

--- Generating Baseline (No Bans) ---
--- Generating Response for model: Qwen/Qwen2.5-1.5B-Instruct ---
Prompt: Describe a beautiful sunset over the ocean. It
Specific Banned Words: []
Random Ban Percentage: 0.00%
Max New Tokens: 100
No tokens will be banned for this generation.
No banning processor needed.
Encoding prompt...
Input tensors moved to device: cuda:0
Generating response with model.generate()...
Generation complete.
Decoding response...
Generated text:  was a perfect day, with the sun setting behind a massive cliff overlooking a vast expanse of blue water. The sky above was painted in shades of orange and pink, with streaks of purple and magenta slicing through it like splashes of paint. As the sun dipped lower towards the horizon, its light began to glow brighter and more intensely.

The waves on the ocean seemed to dance and play as the

In [10]:
print("======================================================================")
print("== Demonstration: Describe an Apple (Banning 'apple') ==")
print("======================================================================")

model_id_for_demo = "Qwen/Qwen2.5-1.5B-Instruct"
prompt_demo = "Describe an apple."
specific_banned_words_demo = ["apple", "apples"," apple", " Apple"] # Ban singular and plural
random_ban_percentage_demo = 0.0
max_new_tokens_demo = 60

model_demo = None
tokenizer_demo = None

try:
    # --- Actual Loading logic for this cell ---
    print(f"DEMO: Loading tokenizer for {model_id_for_demo}...")
    tokenizer_demo = AutoTokenizer.from_pretrained(model_id_for_demo)
    if tokenizer_demo.pad_token is None:
        tokenizer_demo.pad_token = tokenizer_demo.eos_token
        print(f"Set tokenizer.pad_token to tokenizer.eos_token for {model_id_for_demo}")

    model_args_demo = {}
    if "gemma" in model_id_for_demo.lower():
        model_args_demo['torch_dtype'] = torch.float16
        print(f"Note: For Gemma ({model_id_for_demo}), torch_dtype=torch.float16 is set.")
    if "qwen" in model_id_for_demo.lower():
        model_args_demo['trust_remote_code'] = True
        print(f"Note: For Qwen ({model_id_for_demo}), trust_remote_code=True is set.")

    print(f"DEMO: Loading model {model_id_for_demo} with args {model_args_demo}...")
    model_demo = AutoModelForCausalLM.from_pretrained(model_id_for_demo, **model_args_demo)

    if torch.cuda.is_available():
        print(f"Moving model {model_id_for_demo} to CUDA device.")
        model_demo.to('cuda')
    else:
        print(f"CUDA not available for {model_id_for_demo}, using CPU.")
    # --- END Actual Loading logic ---

    print(f"\n--- Scenario Details ---")
    print(f"Model: {model_id_for_demo}")
    print(f"Prompt: {prompt_demo}")
    print(f"Max New Tokens: {max_new_tokens_demo}")

    print(f"\n--- Generating Baseline (No Bans) ---")
    baseline_output_demo = generate_response(
        model_demo,
        tokenizer_demo,
        prompt_demo,
        max_new_tokens=max_new_tokens_demo
    )
    print(f"\nBaseline Output:\n{prompt_demo}{baseline_output_demo}")

    print(f"\n--- Generating with Bans (Banned: {specific_banned_words_demo}) ---")
    banned_output_demo = generate_response(
        model_demo,
        tokenizer_demo,
        prompt_demo,
        specific_banned_words=specific_banned_words_demo,
        random_ban_percentage=random_ban_percentage_demo,
        max_new_tokens=max_new_tokens_demo
    )
    print(f"\nOutput with Bans:\n{prompt_demo}{banned_output_demo}")

except Exception as e:
    print(f"Error during demo scenario '{prompt_demo[:30]}...': {e}")
finally:
    print("\nCleaning up resources for demo scenario...")
    if 'model_demo' in locals() and model_demo is not None: del model_demo
    if 'tokenizer_demo' in locals() and tokenizer_demo is not None: del tokenizer_demo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    print("Demo scenario finished and cleaned up.")
print("======================================================================")


== Demonstration: Describe an Apple (Banning 'apple') ==
DEMO: Loading tokenizer for Qwen/Qwen2.5-1.5B-Instruct...
Note: For Qwen (Qwen/Qwen2.5-1.5B-Instruct), trust_remote_code=True is set.
DEMO: Loading model Qwen/Qwen2.5-1.5B-Instruct with args {'trust_remote_code': True}...
Moving model Qwen/Qwen2.5-1.5B-Instruct to CUDA device.

--- Scenario Details ---
Model: Qwen/Qwen2.5-1.5B-Instruct
Prompt: Describe an apple.
Max New Tokens: 60

--- Generating Baseline (No Bans) ---
--- Generating Response for model: Qwen/Qwen2.5-1.5B-Instruct ---
Prompt: Describe an apple.
Specific Banned Words: []
Random Ban Percentage: 0.00%
Max New Tokens: 60
No tokens will be banned for this generation.
No banning processor needed.
Encoding prompt...
Input tensors moved to device: cuda:0
Generating response with model.generate()...
Generation complete.
Decoding response...
Generated text:  An apple is a fruit that grows on trees in the genus Malus, which includes species such as the domesticated apple (Ma

In [11]:
print("======================================================================")
print("== Demonstration: Describe an Apple (Banning 'apple') ==")
print("======================================================================")

model_id_for_demo = "Qwen/Qwen2.5-1.5B-Instruct"
prompt_demo = "Describe an apple."
specific_banned_words_demo = ["apple", "apples", ' apple', ' Apple'] # Ban singular and plural
random_ban_percentage_demo = 0.0
max_new_tokens_demo = 60

model_demo = None
tokenizer_demo = None

try:
    # --- Actual Loading logic for this cell ---
    print(f"DEMO: Loading tokenizer for {model_id_for_demo}...")
    tokenizer_demo = AutoTokenizer.from_pretrained(model_id_for_demo)
    if tokenizer_demo.pad_token is None:
        tokenizer_demo.pad_token = tokenizer_demo.eos_token
        print(f"Set tokenizer.pad_token to tokenizer.eos_token for {model_id_for_demo}")

    model_args_demo = {}
    if "gemma" in model_id_for_demo.lower():
        model_args_demo['torch_dtype'] = torch.float16
        print(f"Note: For Gemma ({model_id_for_demo}), torch_dtype=torch.float16 is set.")
    if "qwen" in model_id_for_demo.lower():
        model_args_demo['trust_remote_code'] = True
        print(f"Note: For Qwen ({model_id_for_demo}), trust_remote_code=True is set.")

    print(f"DEMO: Loading model {model_id_for_demo} with args {model_args_demo}...")
    model_demo = AutoModelForCausalLM.from_pretrained(model_id_for_demo, **model_args_demo)

    if torch.cuda.is_available():
        print(f"Moving model {model_id_for_demo} to CUDA device.")
        model_demo.to('cuda')
    else:
        print(f"CUDA not available for {model_id_for_demo}, using CPU.")
    # --- END Actual Loading logic ---

    print(f"\n--- Scenario Details ---")
    print(f"Model: {model_id_for_demo}")
    print(f"Prompt: {prompt_demo}")
    print(f"Max New Tokens: {max_new_tokens_demo}")

    print(f"\n--- Generating Baseline (No Bans) ---")
    baseline_output_demo = generate_response(
        model_demo,
        tokenizer_demo,
        prompt_demo,
        max_new_tokens=max_new_tokens_demo
    )
    print(f"\nBaseline Output:\n{prompt_demo}{baseline_output_demo}")

    print(f"\n--- Generating with Bans (Banned: {specific_banned_words_demo}) ---")
    banned_output_demo = generate_response(
        model_demo,
        tokenizer_demo,
        prompt_demo,
        specific_banned_words=specific_banned_words_demo,
        random_ban_percentage=random_ban_percentage_demo,
        max_new_tokens=max_new_tokens_demo
    )
    print(f"\nOutput with Bans:\n{prompt_demo}{banned_output_demo}")

except Exception as e:
    print(f"Error during demo scenario '{prompt_demo[:30]}...': {e}")
finally:
    print("\nCleaning up resources for demo scenario...")
    if 'model_demo' in locals() and model_demo is not None: del model_demo
    if 'tokenizer_demo' in locals() and tokenizer_demo is not None: del tokenizer_demo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    print("Demo scenario finished and cleaned up.")
print("======================================================================")


== Demonstration: Describe an Apple (Banning 'apple') ==
DEMO: Loading tokenizer for Qwen/Qwen2.5-1.5B-Instruct...
Note: For Qwen (Qwen/Qwen2.5-1.5B-Instruct), trust_remote_code=True is set.
DEMO: Loading model Qwen/Qwen2.5-1.5B-Instruct with args {'trust_remote_code': True}...
Moving model Qwen/Qwen2.5-1.5B-Instruct to CUDA device.

--- Scenario Details ---
Model: Qwen/Qwen2.5-1.5B-Instruct
Prompt: Describe an apple.
Max New Tokens: 60

--- Generating Baseline (No Bans) ---
--- Generating Response for model: Qwen/Qwen2.5-1.5B-Instruct ---
Prompt: Describe an apple.
Specific Banned Words: []
Random Ban Percentage: 0.00%
Max New Tokens: 60
No tokens will be banned for this generation.
No banning processor needed.
Encoding prompt...
Input tensors moved to device: cuda:0
Generating response with model.generate()...
Generation complete.
Decoding response...
Generated text:  An apple is a round, juicy fruit that comes in many different colors and sizes. It has a smooth, shiny skin that can 

In [13]:

print("======================================================================")
print("== Demonstration: Math Problem (Banning '+') ==")
print("======================================================================")

model_id_for_demo = "Qwen/Qwen2.5-1.5B-Instruct"
prompt_demo = "What is 2 + 2?"
specific_banned_words_demo = ["+", " +"] # Ban the plus symbol
random_ban_percentage_demo = 0.0
max_new_tokens_demo = 100

model_demo = None
tokenizer_demo = None

try:
    # --- Actual Loading logic for this cell ---
    print(f"DEMO: Loading tokenizer for {model_id_for_demo}...")
    tokenizer_demo = AutoTokenizer.from_pretrained(model_id_for_demo)
    if tokenizer_demo.pad_token is None:
        tokenizer_demo.pad_token = tokenizer_demo.eos_token
        print(f"Set tokenizer.pad_token to tokenizer.eos_token for {model_id_for_demo}")

    model_args_demo = {}
    if "gemma" in model_id_for_demo.lower():
        model_args_demo['torch_dtype'] = torch.float16
        print(f"Note: For Gemma ({model_id_for_demo}), torch_dtype=torch.float16 is set.")
    if "qwen" in model_id_for_demo.lower():
        model_args_demo['trust_remote_code'] = True
        print(f"Note: For Qwen ({model_id_for_demo}), trust_remote_code=True is set.")

    print(f"DEMO: Loading model {model_id_for_demo} with args {model_args_demo}...")
    model_demo = AutoModelForCausalLM.from_pretrained(model_id_for_demo, **model_args_demo)

    if torch.cuda.is_available():
        print(f"Moving model {model_id_for_demo} to CUDA device.")
        model_demo.to('cuda')
    else:
        print(f"CUDA not available for {model_id_for_demo}, using CPU.")
    # --- END Actual Loading logic ---

    print(f"\n--- Scenario Details ---")
    print(f"Model: {model_id_for_demo}")
    print(f"Prompt: {prompt_demo}")
    print(f"Specific Banned Words: {specific_banned_words_demo}")
    print(f"Max New Tokens: {max_new_tokens_demo}")

    print(f"\n--- Generating Baseline (No Bans) ---")
    baseline_output_demo = generate_response(
        model_demo,
        tokenizer_demo,
        prompt_demo,
        max_new_tokens=max_new_tokens_demo
    )
    print(f"\nBaseline Output:\n{prompt_demo}{baseline_output_demo}")

    print(f"\n--- Generating with Bans (Banned: {specific_banned_words_demo}) ---")
    banned_output_demo = generate_response(
        model_demo,
        tokenizer_demo,
        prompt_demo,
        specific_banned_words=specific_banned_words_demo,
        random_ban_percentage=random_ban_percentage_demo,
        max_new_tokens=max_new_tokens_demo
    )
    print(f"\nOutput with Bans:\n{prompt_demo}{banned_output_demo}")

except Exception as e:
    print(f"Error during demo scenario '{prompt_demo[:30]}...': {e}")
finally:
    print("\nCleaning up resources for demo scenario...")
    if 'model_demo' in locals() and model_demo is not None: del model_demo
    if 'tokenizer_demo' in locals() and tokenizer_demo is not None: del tokenizer_demo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    print("Demo scenario finished and cleaned up.")
print("======================================================================")

== Demonstration: Math Problem (Banning '+') ==
DEMO: Loading tokenizer for Qwen/Qwen2.5-1.5B-Instruct...
Note: For Qwen (Qwen/Qwen2.5-1.5B-Instruct), trust_remote_code=True is set.
DEMO: Loading model Qwen/Qwen2.5-1.5B-Instruct with args {'trust_remote_code': True}...
Moving model Qwen/Qwen2.5-1.5B-Instruct to CUDA device.

--- Scenario Details ---
Model: Qwen/Qwen2.5-1.5B-Instruct
Prompt: What is 2 + 2?
Specific Banned Words: ['+', ' +']
Max New Tokens: 100

--- Generating Baseline (No Bans) ---
--- Generating Response for model: Qwen/Qwen2.5-1.5B-Instruct ---
Prompt: What is 2 + 2?
Specific Banned Words: []
Random Ban Percentage: 0.00%
Max New Tokens: 100
No tokens will be banned for this generation.
No banning processor needed.
Encoding prompt...
Input tensors moved to device: cuda:0
Generating response with model.generate()...
Generation complete.
Decoding response...
Generated text:  - Answers\nMath and Arithmetic\nWhat is 2 + 2?\nWiki User\n∙ 2016-08-04 07:39:51\nBest Answer\nCo

In [15]:

print("======================================================================")
print("== Demonstration: 'Awareness' Test - Name Ten Fruits (Ban 'apple') ==")
print("======================================================================")

# Consider using a more capable model if gpt2-large struggles too much with listing.
# model_id_for_demo = "google/gemma-2b-it"
model_id_for_demo = "Qwen/Qwen2.5-1.5B-Instruct"
prompt_demo = "Name ten different types of fruits."
specific_banned_words_demo = ["apple", "apples", " apple", " Apple"]
random_ban_percentage_demo = 0.0
max_new_tokens_demo = 75 # Allow more tokens for a list

model_demo = None
tokenizer_demo = None

try:
    # --- Actual Loading logic for this cell ---
    print(f"DEMO: Loading tokenizer for {model_id_for_demo}...")
    tokenizer_demo = AutoTokenizer.from_pretrained(model_id_for_demo)
    if tokenizer_demo.pad_token is None:
        tokenizer_demo.pad_token = tokenizer_demo.eos_token
        print(f"Set tokenizer.pad_token to tokenizer.eos_token for {model_id_for_demo}")

    model_args_demo = {}
    if "gemma" in model_id_for_demo.lower():
        model_args_demo['torch_dtype'] = torch.float16
        print(f"Note: For Gemma ({model_id_for_demo}), torch_dtype=torch.float16 is set.")
    if "qwen" in model_id_for_demo.lower():
        model_args_demo['trust_remote_code'] = True
        print(f"Note: For Qwen ({model_id_for_demo}), trust_remote_code=True is set.")

    print(f"DEMO: Loading model {model_id_for_demo} with args {model_args_demo}...")
    model_demo = AutoModelForCausalLM.from_pretrained(model_id_for_demo, **model_args_demo)

    if torch.cuda.is_available():
        print(f"Moving model {model_id_for_demo} to CUDA device.")
        model_demo.to('cuda')
    else:
        print(f"CUDA not available for {model_id_for_demo}, using CPU.")
    # --- END Actual Loading logic ---

    print(f"\n--- Scenario Details ---")
    print(f"Model: {model_id_for_demo}")
    print(f"Prompt: {prompt_demo}")
    print(f"Specific Banned Words: {specific_banned_words_demo}")
    print(f"Max New Tokens: {max_new_tokens_demo}")

    print(f"\n--- Generating Baseline (No Bans) ---")
    baseline_output_demo = generate_response(
        model_demo,
        tokenizer_demo,
        prompt_demo,
        max_new_tokens=max_new_tokens_demo
    )
    print(f"\nBaseline Output:\n{prompt_demo}{baseline_output_demo}")

    print(f"\n--- Generating with Bans (Banned: {specific_banned_words_demo}) ---")
    banned_output_demo = generate_response(
        model_demo,
        tokenizer_demo,
        prompt_demo,
        specific_banned_words=specific_banned_words_demo,
        random_ban_percentage=random_ban_percentage_demo,
        max_new_tokens=max_new_tokens_demo
    )
    print(f"\nOutput with Bans:\n{prompt_demo}{banned_output_demo}")
    print("\nAnalyze: Is 'apple' (and 'apples') omitted? Does the model struggle or list fewer valid fruits?")

except Exception as e:
    print(f"Error during demo scenario '{prompt_demo[:30]}...': {e}")
finally:
    print("\nCleaning up resources for demo scenario...")
    if 'model_demo' in locals() and model_demo is not None: del model_demo
    if 'tokenizer_demo' in locals() and tokenizer_demo is not None: del tokenizer_demo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    print("Demo scenario finished and cleaned up.")
print("======================================================================")

== Demonstration: 'Awareness' Test - Name Ten Fruits (Ban 'apple') ==
DEMO: Loading tokenizer for Qwen/Qwen2.5-1.5B-Instruct...
Note: For Qwen (Qwen/Qwen2.5-1.5B-Instruct), trust_remote_code=True is set.
DEMO: Loading model Qwen/Qwen2.5-1.5B-Instruct with args {'trust_remote_code': True}...
Moving model Qwen/Qwen2.5-1.5B-Instruct to CUDA device.

--- Scenario Details ---
Model: Qwen/Qwen2.5-1.5B-Instruct
Prompt: Name ten different types of fruits.
Specific Banned Words: ['apple', 'apples', ' apple', ' Apple']
Max New Tokens: 75

--- Generating Baseline (No Bans) ---
--- Generating Response for model: Qwen/Qwen2.5-1.5B-Instruct ---
Prompt: Name ten different types of fruits.
Specific Banned Words: []
Random Ban Percentage: 0.00%
Max New Tokens: 75
No tokens will be banned for this generation.
No banning processor needed.
Encoding prompt...
Input tensors moved to device: cuda:0
Generating response with model.generate()...
Generation complete.
Decoding response...
Generated text:  Apple, 

In [18]:
print("======================================================================")
print("== Demonstration: Random Ban Test - Short Story ==")
print("======================================================================")

model_id_for_demo = "Qwen/Qwen2.5-1.5B-Instruct"
prompt_demo = "Tell me a short story about a friendly robot who discovered a hidden garden."
max_new_tokens_demo = 150
# Test a range of random ban percentages, including 0% as a baseline
# random_ban_percentages_to_test = [0.0, 0.01, 0.05, 0.10, 0.20]
random_ban_percentages_to_test = [0.75, 0.9]

model_demo = None
tokenizer_demo = None

try:
    # --- Actual Loading logic for this cell ---
    print(f"DEMO: Loading tokenizer for {model_id_for_demo}...")
    tokenizer_demo = AutoTokenizer.from_pretrained(model_id_for_demo)
    if tokenizer_demo.pad_token is None:
        tokenizer_demo.pad_token = tokenizer_demo.eos_token
        print(f"Set tokenizer.pad_token to tokenizer.eos_token for {model_id_for_demo}")

    model_args_demo = {}
    if "gemma" in model_id_for_demo.lower():
        model_args_demo['torch_dtype'] = torch.float16
        print(f"Note: For Gemma ({model_id_for_demo}), torch_dtype=torch.float16 is set.")
    if "qwen" in model_id_for_demo.lower():
        model_args_demo['trust_remote_code'] = True
        print(f"Note: For Qwen ({model_id_for_demo}), trust_remote_code=True is set.")

    print(f"DEMO: Loading model {model_id_for_demo} with args {model_args_demo}...")
    model_demo = AutoModelForCausalLM.from_pretrained(model_id_for_demo, **model_args_demo)

    if torch.cuda.is_available():
        print(f"Moving model {model_id_for_demo} to CUDA device.")
        model_demo.to('cuda')
    else:
        print(f"CUDA not available for {model_id_for_demo}, using CPU.")
    # --- END Actual Loading logic ---

    print(f"\n--- Scenario Details ---")
    print(f"Model: {model_id_for_demo}")
    print(f"Prompt: {prompt_demo}")
    print(f"Max New Tokens: {max_new_tokens_demo}")

    for pc_idx, percentage in enumerate(random_ban_percentages_to_test):
        print(f"\n--- Test {pc_idx+1}: Random Ban Percentage: {percentage*100:.2f}% ---")
        # For random ban, specific_banned_words is None or empty list
        output_demo = generate_response(
            model_demo,
            tokenizer_demo,
            prompt_demo,
            specific_banned_words=[], # Ensure no specific bans for this test
            random_ban_percentage=percentage,
            max_new_tokens=max_new_tokens_demo
        )
        print(f"Output (Random Ban {percentage*100:.2f}%):\n{prompt_demo}{output_demo}")
        if percentage > 0.05: # Arbitrary threshold for analysis prompt
             print("\nAnalyze: Observe output degradation, coherence, use of unusual words, or other notable changes as random ban % increases.")

except Exception as e:
    print(f"Error during demo scenario '{prompt_demo[:30]}...': {e}")
finally:
    print("\nCleaning up resources for demo scenario...")
    if 'model_demo' in locals() and model_demo is not None: del model_demo
    if 'tokenizer_demo' in locals() and tokenizer_demo is not None: del tokenizer_demo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

== Demonstration: Random Ban Test - Short Story ==
DEMO: Loading tokenizer for Qwen/Qwen2.5-1.5B-Instruct...
Note: For Qwen (Qwen/Qwen2.5-1.5B-Instruct), trust_remote_code=True is set.
DEMO: Loading model Qwen/Qwen2.5-1.5B-Instruct with args {'trust_remote_code': True}...
Moving model Qwen/Qwen2.5-1.5B-Instruct to CUDA device.

--- Scenario Details ---
Model: Qwen/Qwen2.5-1.5B-Instruct
Prompt: Tell me a short story about a friendly robot who discovered a hidden garden.
Max New Tokens: 150

--- Test 1: Random Ban Percentage: 75.00% ---
--- Generating Response for model: Qwen/Qwen2.5-1.5B-Instruct ---
Prompt: Tell me a short story about a friendly robot who discovered a hidden garden.
Specific Banned Words: []
Random Ban Percentage: 75.00%
Max New Tokens: 150
Randomly banning 113732 tokens from 151643 eligible tokens (vocab size: 151643).
Final list of 113732 unique token ID(s) to ban: [0, 1, 2, 3, 6, 7, 8, 9, 10, 11, 13, 14, 16, 17, 18, 20, 22, 23, 25, 26]... (truncated if >20)
TokenBan

In [19]:
tokenizer_demo = AutoTokenizer.from_pretrained(model_id_for_demo)

In [ ]:
s

In [20]:
tokenizer_demo.encode("has been")

[4648, 1012]

In [23]:
tokenizer_demo.encode("had always been")

[31245, 2677, 1012]